# Working with Files in Python & Pandas

In this notebook, we practice the fundamental file I/O operations required for tabular machine learning workflows:
- Loading CSV files into Pandas DataFrames using `pd.read_csv()`
- Using important parameters such as `usecols` and `nrows`
- Inspecting data safely with `head()`, `tail()`, and unbiased random sampling with `sample()`
- Exporting processed data to CSV using `df.to_csv(index=False)`

---


## 1. Import Libraries

We start by importing Pandas. This is the primary library used for tabular data manipulation in Python.


In [1]:
import pandas as pd
import os

# Confirm the dataset path exists
data_path = '../../data/titanic.csv'
print(f"Data file exists: {os.path.exists(data_path)}")


Data file exists: True


## 2. Reading CSV Files (`pd.read_csv`)

A CSV (Comma-Separated Values) file stores rows of records where fields are separated by commas.
`pd.read_csv()` reads the file and automatically converts it into a 2-dimensional tabular structure called a **DataFrame**.


In [2]:
# Load the Titanic dataset
df = pd.read_csv('../../data/titanic.csv')

# Verify the type of object created
type(df)


pandas.core.frame.DataFrame

The file has been loaded into a `pandas.core.frame.DataFrame`. We can now interact with it using Pandas functions and methods.


## 3. Inspecting Rows from the Beginning (`head`)

`head()` displays the first $n$ rows of the DataFrame (defaults to 5 if no argument is passed).

It allows us to quickly verify:
- Did column headers parse correctly?
- Do the values match the expected format?


In [3]:
# Inspect the first 5 rows
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Observation
The first 5 rows confirm our column headers (`PassengerId`, `Survived`, `Pclass`, `Name`, `Sex`, `Age`, etc.).
We can already spot numerical features (e.g., `Age`, `Fare`), categorical features (e.g., `Sex`, `Embarked`), and missing values (e.g., `NaN` in `Cabin`).


We can pass an integer argument to view any number of rows.


In [4]:
# Check only the first 3 rows
df.head(3)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 4. Inspecting Rows from the End (`tail`)

`tail()` displays the last $n$ rows of the DataFrame.

This is especially helpful for catching:
- Corrupted trailing lines in the file
- Unwanted total/summary rows often found in Excel-exported CSVs


In [5]:
# Check the last 5 rows of the dataset
df.tail()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


### Observation
The last passenger index is 890 (total of 891 rows). The data ends cleanly with regular passenger records rather than summary statistics or trailing delimiters.


## 5. Unbiased Inspection with Random Sampling (`sample`)

Why not just rely on `head()` and `tail()`?
In many real-world datasets, rows are ordered by class, date, or target value. Inspecting only the top or bottom can create a false impression of the data.

`sample(n, random_state=...)` randomly selects rows across the entire dataset.
Setting `random_state` ensures reproducibility so that we see the exact same sample every time.


In [6]:
# Take a random sample of 5 rows with a fixed seed for reproducibility
df.sample(5, random_state=42)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
709,710,1,3,"Moubarek, Master. Halim Gonios (""William George"")",male,NaN,1,1,2661,15.2458,NaN,C
439,440,0,2,"Kvillner, Mr. Johan Henrik Johannesson",male,31.0,0,0,C.A. 18723,10.5000,NaN,S
840,841,0,3,"Alhomaki, Mr. Ilmari Rudolf",male,20.0,0,0,SOTON/O2 3101287,7.9250,NaN,S
720,721,1,2,"Harper, Miss. Annie Jessie ""Nina""",female,6.0,0,1,248727,33.0000,NaN,S
39,40,1,3,"Nicola-Yarred, Miss. Jamila",female,14.0,1,0,2651,11.2417,NaN,C


### Observation
The sampled rows come from varied indices (709, 439, 840, 720, 39). Notice the mix of passenger classes (1st, 2nd, and 3rd) and genders. Random sampling provides a much more honest initial glimpse into data variance.


## 6. Memory Optimization & Selective Reading

When working with very large datasets, loading every column or all million rows at once can exhaust available RAM.
`pd.read_csv()` provides parameters to read selectively:
- `usecols`: Loads only the specified list of columns.
- `nrows`: Reads only the first $k$ rows.


In [7]:
# Load only a subset of relevant columns
selected_cols = ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'Fare']
df_subset = pd.read_csv('../../data/titanic.csv', usecols=selected_cols, nrows=10)

# Check the subset shape and preview
print('Subset shape:', df_subset.shape)
df_subset


Subset shape: (10, 6)


,PassengerId,Survived,Pclass,Sex,Age,Fare
0,1,0,3,male,22.0,7.2500
1,2,1,1,female,38.0,71.2833
2,3,1,3,female,26.0,7.9250
3,4,1,1,female,35.0,53.1000
4,5,0,3,male,35.0,8.0500
5,6,0,3,male,NaN,8.4583
6,7,0,1,male,54.0,51.8625
7,8,0,3,male,2.0,21.0750
8,9,1,3,female,27.0,11.1333
9,10,1,2,female,14.0,30.0708


### Observation
By specifying `usecols` and `nrows`, Pandas only parses 6 columns and 10 rows. This is an essential technique for exploring gigabyte-scale datasets without memory overflow.


## 7. Writing Data to CSV (`to_csv`)

Once we filter, clean, or engineer features, we often save the resulting DataFrame for downstream modeling.

**Critical Rule:** Always pass `index=False` unless the numerical index represents an explicit domain identifier.
If you omit `index=False`, Pandas will save an unnamed index column (`Unnamed: 0`), cluttering future reads.


In [8]:
# Filter female passengers traveling in First Class who survived
first_class_females_survived = df[(df['Pclass'] == 1) & (df['Sex'] == 'female') & (df['Survived'] == 1)]

# Save the filtered subset to disk
output_file = '../../outputs/titanic_first_class_female_survivors.csv'
first_class_females_survived.to_csv(output_file, index=False)

print(f"Saved {len(first_class_females_survived)} records to {output_file}")


Saved 91 records to ../../outputs/titanic_first_class_female_survivors.csv


### Verifying the Exported File
Let us read the exported CSV back to ensure there is no unwanted `Unnamed: 0` column.


In [9]:
# Reload the newly saved file
df_reloaded = pd.read_csv('../../outputs/titanic_first_class_female_survivors.csv')

# Verify columns and check first 3 rows
print('Reloaded columns:', df_reloaded.columns.tolist())
df_reloaded.head(3)


Reloaded columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
1,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
2,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S


### Key Takeaways
1. Use `pd.read_csv()` to load flat files, utilizing `usecols` and `nrows` for memory efficiency.
2. Combine `head()`, `tail()`, and `sample(random_state=42)` to inspect the dataset without sampling bias.
3. Always export with `index=False` to maintain clean tabular files.
